# Tutorial for Hybrid Modeling of Entanglement Distribution Sources

In this tutorial, we will explore the complete process of modeling entanglement distribution sources using a hybrid Gaussian/non-Gaussian modeling approach. The fundamental theory for this approach can be found in (the accompanying paper that this code will be published with). To being, let's call the functions needed for modeling. ?

In [1]:
using Genqo
using LinearAlgebra
using Plots

ErrorException: Error opening package file C:\Users\jomav\.julia\compiled\v1.12\Preferences\pWSk8_XAGxp.dll: An Application Control policy has blocked this file. 


Matrix sparcity

In [ ]:

function sparcity_ratio(μ::Real, ηᵗ::Real, ηᵈ::Real, ηᵇ::Real, dark_counts::Real)
    cov = Genqo.zalm.covariance_matrix(μ)
    A = Genqo.tools.k_function_matrix(cov) + Genqo.zalm.loss_bsm_matrix_pgen(ηᵗ, ηᵈ, ηᵇ)
    Ainv = inv(A)
    println("Ainv: ", size(Ainv,1))
   return count(!iszero, Ainv) / length(Ainv)
end
 sparcity_ratio(1e-2,1,1,1,1)

Recursive algorithm

In [ ]:

function recursive_algorithm(A::Matrix{ComplexF64})
    nb_lines = size(A,1)

    if nb_lines % 2 != 0
        return 0
    end

    n = nb_lines ÷ 2

    z = zeros(ComplexF64, n * (2n - 1), n + 1)

    for j in 1:(2n - 1)
        ind = j * (j - 1) ÷ 2
        for k in 0:(j - 1)
            z[ind + k + 1, 1] = A[j + 1, k + 1]
        end
    end

    g = zeros(ComplexF64, n + 1)
    g[1] = one(ComplexF64)

    return solve_recursive(z, 2n, 1, g, n)
end


function solve_recursive(
    b::Matrix{ComplexF64},
    s::Int,
    w::Int,
    g::AbstractVector{ComplexF64},
    n::Int,
)

    if s == 0
        return w * g[n + 1]
    end

    c = zeros(ComplexF64, ((s - 2) * (s - 3)) ÷ 2, n + 1)

    i = 1

    for j in 1:(s - 3)
        for k in 0:(j - 1)
            src_row = ((j + 1) * (j + 2)) ÷ 2 + k + 3
            c[i, :] .= b[src_row, :]
            i += 1
        end
    end

    h = solve_recursive(c, s - 2, -w, g, n)

    e = copy(g)

    for u in 0:(n - 1)
        for v in 0:(n - u - 1)
            e[u + v + 2] += g[u + 1] * b[1, v + 1]
        end
    end

    for j in 1:(s - 3)
        for k in 0:(j - 1)
            c_row = j * (j - 1) ÷ 2 + k + 1

            for u in 0:(n - 1)
                for v in 0:(n - u - 1)
                    c[c_row, u + v + 2] +=
                        b[((j + 1) * (j + 2)) ÷ 2 + 1, u + 1] *
                        b[((k + 1) * (k + 2)) ÷ 2 + 2, v + 1] +
                        b[((k + 1) * (k + 2)) ÷ 2 + 1, u + 1] *
                        b[((j + 1) * (j + 2)) ÷ 2 + 2, v + 1]
                end
            end
        end
    end

    return h + solve_recursive(c, s - 2, w, e, n)
end

In [ ]:
function hafnian_sparse(A::AbstractMatrix{ComplexF64}, D::Set{Int}=nothing; loop::Bool=false)
    n = size(A, 1)

    Dset = D === nothing ? Set(1:n) : Set(D)

    B = copy(A)
    if !loop
        for i in 1:n
            B[i, i] = zero(ComplexF64)
        end
    end

    # Python: if np.allclose(A, 0): return 0.0
    if all(x -> isapprox(x, zero(ComplexF64); atol=1e-12, rtol=1e-12), B)
        return zero(ComplexF64)
    end

    # Memoization cache.
    # Use sorted tuples as keys because Set is mutable and unsafe as a Dict key.
    cache = Dict{Tuple{Vararg{Int}}, ComplexF64}()

    function lhaf(d::Set{Int})::ComplexF64
        key = Tuple(sort(collect(d)))

        if haskey(cache, key)
            return cache[key]
        end

        if isempty(d)
            return one(T)
        end

        d_without_k = copy(d)
        k = pop!(d_without_k)

        # Python: indices(d, k) = d ∩ nonzero(A[k, :])
        nonzero_cols = findall(j -> !iszero(B[k, j]), 1:n)
        js = intersect(d, Set(nonzero_cols))

        result = zero(T)
        for j in js
            next_d = setdiff(d_without_k, Set([j]))
            result += B[j, k] * lhaf(next_d)
        end

        cache[key] = result
        return result
    end

    return lhaf(Dset)
end

In [ ]:
function Ainv(μ::Real, ηᵗ::Real, ηᵈ::Real, ηᵇ::Real, dark_counts::Real)
    cov = Genqo.zalm.covariance_matrix(μ)
    A = Genqo.tools.k_function_matrix(cov) + Genqo.zalm.loss_bsm_matrix_pgen(ηᵗ, ηᵈ, ηᵇ)
    return inv(A)
end

A = Ainv(1e-2,1,1,1,1)
recursive_algorithm(A)

Benchmark


In [ ]:
using BenchmarkTools
using Random
using Statistics

In [ ]:

function random_complex_symmetric_matrix(n::Int)
    A = randn(ComplexF64, n, n)
    return (A + transpose(A)) / 2
end
function benchmark_hafnian_sizes(sizes::Vector{Int}; samples::Int = 10)
    times = Float64[]

    for N in sizes
        println("Benchmarking matrix size $N x $N")
        A = random_complex_symmetric_matrix(N)
        recursive_algorithm(A)
        bench = @benchmark recursive_algorithm($A) samples=samples evals=1
        push!(times, median(bench.times) / 1e9)
    end

    return times
end

sizes = collect(2:2:24) 
times = benchmark_hafnian_sizes(sizes; samples=100)

plot(
    sizes,times,marker=:circle, xlabel="Matrix size N", ylabel="Median runtime, seconds",title="Recursive Hafnian Benchmark",label="recursive_algorithm", legend=:bottomright, yscale=:log10,
)